# Checkpoint Two: Exploratory Data Analysis

Now that your chosen dataset is approved, it is time to start working on your analysis. Use this notebook to perform your EDA and make notes where directed to as you work.

## Getting Started

Since we have not provided your dataset for you, you will need to load the necessary files in this repository. Make sure to include a link back to the original dataset here as well.

My dataset:https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis/data

Your first task in EDA is to import necessary libraries and create a dataframe(s). Make note in the form of code comments of what your thought process is as you work on this setup task.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
from matplotlib import style

: 

In [ ]:
df = pd.read_csv(r"C:\Users\pered\OneDrive\Documents\Launch code\Data Analysis\Final Project folder\Datco Supply chain dataset\SupplyChainDataset.csv",
                encoding='latin1')

In [ ]:
print(df.info())

## Get to Know the Numbers

Now that you have everything setup, put any code that you use to get to know the dataframe and its rows and columns better in the cell below. You can use whatever techniques you like, except for visualizations. You will put those in a separate section.

When working on your code, make sure to leave comments so that your mentors can understand your thought process.

In [ ]:
df.head(5)

In [ ]:
df.tail(5)

In [ ]:
df.shape

In [ ]:
## to get look at null counts:

df.isnull().sum()

Exploratory Data Analysis:

I will perform Exploratory Data Analysis (EDA) to explore the dataset, identify which columns are important for analysis, and understand the factors that influence the key variables. 

The goal is to gain insights into the data before moving on to further steps:

Supply Chain View:

In the Supply Chain View,the goal is to understand delivery performance,shipping delays, and the factors that influence late deliveries.

The following analyses were performed:



In [ ]:
df.columns

In [ ]:
## Create a delay column
df['Delay_Days'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']

In [ ]:
df

1)Average Delay per Shipping Mode

I calculated the average number of days delayed for each shipping mode to understand which modes are generally slower or faster. 
This helps identify underperforming delivery methods.

In [ ]:
## check for the unique shipping modes:
df['Shipping Mode'].unique()

In [ ]:
df.groupby("Shipping Mode")["Delay_Days"].mean().sort_values(ascending=False)

2)Count of Delayed Orders per Shipping Mode

I counted how many orders were delivered late for each shipping mode. This shows which shipping method contributes the most to overall delays.

In [ ]:
df[df['Delay_Days'] > 0].groupby("Shipping Mode").size().sort_values(ascending=False)

3)Regions With the Worst Delivery Performance

I calculated the mean delay grouped by:

Order Region

Market

This identifies which regions experience the highest delivery delays and may need operational improvement.

In [ ]:
region_delay = (
    df.groupby("Order Region")["Late_delivery_risk"]
      .mean()
      .round(3)
      .sort_values(ascending=False) * 100
)

region_delay

In [ ]:
## Delay orders by Market:
market_delay = (
    df.groupby("Market")["Late_delivery_risk"]
      .mean()
      .sort_values(ascending=False) * 100
)

market_delay

4)Count of Late Orders by Delivery Status

I compared the number of orders across different delivery status to understand how delays are distributed (e.g., Late, On Time, Advance, etc.).

In [ ]:
df['Delivery Status'].value_counts()

5)Late Delivery Risk Analysis by Multiple Factors

I performed a deeper risk analysis by grouping delays across multiple key dimensions:

Category Name

Customer Segment

In [ ]:
# % of delayed orders by category_name: 
Late_delivery_risk_by_Category_Name = (
    df
    .groupby("Category Name")["Late_delivery_risk"]
    .mean()
    .mul(100)                       
    .round(2)                       
    .reset_index(name='Late Delivery %') 
    .sort_values("Late Delivery %", ascending=False)
)
Late_delivery_risk_by_Category_Name["Late Delivery %"] = (
    Late_delivery_risk_by_Category_Name["Late Delivery %"].astype(str) + "%"
)

Late_delivery_risk_by_Category_Name

In [ ]:
# % of delayed orders by customer_segment:
Late_delivery_risk_by_Customer_Segment = (
    df
    .groupby("Customer Segment")["Late_delivery_risk"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name='Late Delivery %')
    .sort_values("Late Delivery %", ascending=False)
)
Late_delivery_risk_by_Customer_Segment

Summary of Supply Chain EDA:

Through these analyses, I examined delays from different angles—shipping mode, region, product category, and customer segment. This helps uncover where delays are happening, why they occur, and which areas need improvement in the supply chain.

Finance View:

The Finance View focuses on understanding how delivery performance and product factors influence profitability. 

The goal is to analyze how delays, shipping choices, and product categories affect financial outcomes. 

The following analyses were performed:

In [ ]:
df['Delayed_Flag'] = df['Delay_Days'].apply(lambda x: 1 if x > 0 else 0)

In [ ]:
df_delayed = df[df['Delayed_Flag'] == 1]

In [ ]:
total_profit_impact_m = float(df_delayed['Sales'].sum().round(2)) / 1_000_000
print(f"{total_profit_impact_m:.2f}M")

In [ ]:
## Profit impact from delayed orders
df['Profit_Impact'] = df['Order Profit Per Order']*df['Delayed_Flag']

1)Profit Impact by Shipping Mode

I analyzed the total and average profit generated by each shipping mode.

This helps identify:

1)Which shipping mode leads to higher or lower profit.


In [ ]:
## Profit Impact by Shipping Mode:
profit_impact_by_Shipping_mode = (
         df.groupby('Shipping Mode')['Profit_Impact']
         .sum()
         .div(10000)
         .round(2)
         .sort_values(ascending=False)
         .reset_index(name ='Profit_Impact')
)
profit_impact_by_Shipping_mode['Profit_Impact'] = (
               profit_impact_by_Shipping_mode['Profit_Impact'].astype(str) + "M"
)
profit_impact_by_Shipping_mode

2)Profit Impact by Delivery Status

Compared profit across different delivery statuses (On Time, Late, Advance, etc.).

This shows how late deliveries affect profit, and whether timely fulfillment results in higher profitability.

In [ ]:
## Profit from Early / On-Time / Late Deliveries
Profit_by_delivery_status = (
                  df.groupby('Delivery Status')['Order Profit Per Order']
                    .sum()
                    .div(10000)
                    .round(2) 
                    .sort_values(ascending = False)
                    .reset_index(name='Order Profit Per Order')
)
Profit_by_delivery_status['Order Profit Per Order'] = (
               Profit_by_delivery_status['Order Profit Per Order'].astype(str) + "M"
)
Profit_by_delivery_status

: 

3)Profit Impact for Top 10 Products Affected by Delayed Orders

I found the top 10 products most affected by delivery delays and calculated their profit impact.
This shows:

Which products lose the most profit due to delays

Whether high-value items are more sensitive to delivery performance

In [ ]:
## Profit impact by Category:
Profit_Impact_by_Category = (
           df.groupby('Category Name')['Profit_Impact']
             .sum()
             .div(10000)
             .round(2)
             .sort_values(ascending = False)
             .reset_index(name = 'Profit_Impact')
)
Profit_Impact_by_Category['Profit_Impact'] = (
                     Product_Impact_by_Category['Profit_Impact'].astype(str) + "M"
)
Profit_Impact_by_Category

In [ ]:
## Top 10 products profit impact by the delayed orders
top_10_products = Profit_Impact_by_Category.head(10)
top_10_products

4)Profit Impact for Top 10 States With Delayed Orders

I identified the top 10 states with the highest number of delayed orders and analyzed their profit impact.
This helps identify regions where delays may be:

Reducing profit

Increasing operational cost

Affecting customer satisfaction and repeat business

In [ ]:
## Top 10 states causing highest profit loss
profit_impact_by_order_state = (
         df.groupby('Order State')['Profit_Impact']
         .sum()
         .div(10000)
         .round(2)
         .sort_values(ascending=False)
         .reset_index(name ='Profit_Impact')
)
profit_impact_by_order_state['Profit_Impact'] = (
               profit_impact_by_order_state['Profit_Impact'].astype(str) + "M"
)
profit_impact_by_order_state

In [ ]:
## Top 10 states profit impact by the delayed orders
top_10_states = profit_impact_by_order_state.head(10)
top_10_states

Finding the profit impact by the year,so i need to extract the year from the order date columns.So I followed these steps
1)Renaming the order date
2)Normalizing the order date
3)Converts the order date column to a datetime type
4)Extracted the year from the “order date” column and creates a new column called Order_Year.

In [ ]:
## Renaming the order date
df = df.rename(columns={'order date (DateOrders)':'order date'})

In [ ]:
#Normalizing the Order date
df['order date'] = df['order date'].str.strip().replace("",np.nan)

In [ ]:
# Converts the order date column to a datetime type
df['order date'] = pd.to_datetime(df['order date'],errors='coerce')

In [ ]:
# extracts the year from the “order date” column and creates a new column called Order_Year.
df['Order_Year'] = df['order date'].dt.year

In [ ]:
#Profit impact by year:
profit_impact_yearly = (
        df.groupby('Order_Year')['Profit_Impact']
          .sum()
          .div(1000000)
          .round(3)
          .reset_index(name='Profit_Impact')
         
)
profit_impact_yearly['Profit_Impact'] = (
         profit_impact_yearly['Profit_Impact'].astype(str) + "M"
)
profit_impact_yearly

Summary of Finance EDA:

Through these analyses, I explored how logistics and product features influence profitability.
The Finance View provides insights into:

How delays reduce profit

Which regions and products suffer the most financially

Which shipping modes and product categories drive profit

This information helps guide decisions to improve both supply chain performance and financial outcomes.

Sales View:

The Sales View focuses on understanding customer purchasing behavior, product demand, and sales trends across different dimensions. The goal is to identify which factors drive sales performance and how delivery outcomes influence sales.

1)Sales by Delivery Status

I analyzed total sales across different delivery statuses (On Time, Late, Advance, etc.).
This helps reveal:

Whether late deliveries affect sales volume

If customers who receive late orders place fewer or lower-value orders

How delivery performance relates to revenue

In [ ]:
def status(x):
    if x < 0:
        return "Early"
    elif x == 0:
        return "On Time"
    else:
        return "Late"

df['Delivery_Status'] = df['Delay_Days'].apply(status)

In [ ]:
## sales by delivery status
Sales_By_Delivery_Status = (
            df.groupby('Delivery_Status')['Sales']
            .sum()
            .sort_values(ascending = False)
            .reset_index(name='Sales')
)
Sales_By_Delivery_Status['Sales'] = (
                (Sales_By_Delivery_Status['Sales']/1000000)
                     .round(2)
                     .astype(str) + "M"
)
Sales_By_Delivery_Status

2)Average Order Value (AOV)

I calculated the average order value to understand the typical revenue generated per order.
This helps measure:

Customer spending behavior

The value contribution of each order

Differences in AOV across segments or categories

In [ ]:
total_sales = float(df['Sales'].sum().round(2)) / 1_000_000
print(f"{total_sales:.2f}M")

In [ ]:
total_orders = float(df['Order Id'].sum().round(3))/1_000_000
print(f"{total_orders:.2f}M")

In [ ]:
AOV = total_sales/total_orders
AOV

3)Sales Trend by Year

I examined how sales changed over time by grouping orders by year.
This helps identify:

Growth or decline in sales

Seasonal patterns

Demand changes across multiple years

In [ ]:
## Sales trend by year
Sales_Trend_By_Year = (
              df.groupby(['Order_Year','Delivery_Status'])['Sales']
                .sum()
                .sort_values(ascending = False)
                .reset_index()
)
Sales_Trend_By_Year['Sales'] = (
                (Sales_Trend_By_Year['Sales']/1000000)
                     .round(2)
                     .astype(str) + "M"
)
Sales_Trend_By_Year

4)Sales Trend by Category

I explored sales trends over time within each category.
This helps identify:

Which categories are growing or declining

Demand patterns for different product types

Categories affected by seasonal or yearly changes

In [ ]:
#Sales by Category
Sales_Trend_By_Category = (
              df[df['Delivery_Status'] == 'Late']
                 .groupby('Category Name')['Sales']
                .sum()
                .sort_values(ascending = False)
                .reset_index()
)
Sales_Trend_By_Category['Sales'] = (
                (Sales_Trend_By_Category['Sales']/1000000)
                     .round(5)
                     .astype(str) + "M"
)
Sales_Trend_By_Category

6)Regional-Level Sales

I analyzed sales across different regions (countries, states, or markets).
This helps reveal:

Top-performing regions

Regions with low sales

Geographic patterns that influence demand

In [ ]:
## Region-level Sales
Sales_Trend_By_Region = (
              df[df['Delivery_Status'] == 'Late']
                 .groupby('Order Region')['Sales']
                .sum()
                .sort_values(ascending = False)
                .reset_index()
)
Sales_Trend_By_Region['Sales'] = (
                (Sales_Trend_By_Region['Sales']/1000000)
                     .round(2)
                     .astype(str) + "M"
)
Sales_Trend_By_Region

7)Segment-Level Sales

I explored sales across different customer segments (Consumer, Corporate, Home Office, etc.).
This helps identify:

Which customer segment buys the most

Revenue contribution of each segment

Opportunities to target high-value customer groups.

In [ ]:
## Segment-level Sales
Sales_Trend_By_Customer_Segment = (
              df[df['Delivery_Status'] == 'Late']
                  .groupby('Customer Segment')['Sales']
                  .sum()
                  .reset_index()
)
Sales_Trend_By_Customer_Segment['Sales'] = (
                    (Sales_Trend_By_Customer_Segment['Sales']/1000000)
                      .round(1)
                      .astype(str) + "M"
)
Sales_Trend_By_Customer_Segment

Summary of Sales EDA:

The Sales View helps understand the business from a revenue perspective.
It highlights:

How delivery status impacts sales

Customer spending patterns

Yearly and category-wise sales trends

Geographic and segment-level performance

These insights support better decision-making in marketing, inventory planning, and forecasting.

## Visualize

Create any visualizations for your EDA here. Make note in the form of code comments of what your thought process is for your visualizations.

Supply Chain Visualizations:

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Late delivery % by region
plt.figure(figsize=(10,5))
region_delay = df.groupby('Order Region')['Late_delivery_risk'].mean().reset_index()
sns.barplot(data=region_delay, x='Order Region', y='Late_delivery_risk')
plt.title("Late Delivery % by Region")
plt.xticks(rotation=45)
plt.ylabel("Late Delivery Percentage")
plt.show()

# Thought Process:
# Different regions may have different performance levels.
# This chart shows which regions struggle the most with delays.


Finance View Visualizations:

In [ ]:
# 1. Profit by delivery status
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='Delivery Status', y='Profit_Impact')
plt.title("Profit Impact Distribution by Delivery Status")
plt.ylabel("Profit Impact")
plt.show()

# Thought Process:
# Late deliveries may cause refunds, discounts, or lost repeat business.
# This chart visually shows which delivery statuses impact profit.

Sales View Visualizations:

In [ ]:
# 1. Sales by delivery status
# 1. Sales by delivery status (Pie Chart)
sales_by_status = df.groupby('Delivery Status')['Sales'].sum()

plt.figure(figsize=(8,6))
plt.pie(
    sales_by_status,
    labels=sales_by_status.index,
    autopct='%1.1f%%',
    startangle=90
)

plt.title("Sales Distribution by Delivery Status")
plt.axis('equal')   # Makes the pie chart a perfect circle
plt.show()

# Thought Process:
# If late deliveries lead to lower sales, this chart will reveal it.
# Helps determine if customer satisfaction is impacted.

In my EDA visualizations,I created three views-Supply Chain,Finance,and Sales.

The goal was to show how delayed orders affect business performance in different ways.

Supply Chain View -> Shows operational impact(delay patterns,regions,shipping modes).

Finance View -> Shows profit impact(loss due to delys,affected regions/products).

Sales View -> Shows business/customer impact(Sales drop,segment/category efffects).

These visualizations together provide a 360-degree understanding of how late deliveries harm the overall business.

## Summarize Your Results

With your EDA complete, answer the following questions.

1.Was there anything surprising about your dataset? 

 I noticed that the Customer State and Customer Country columns contain names that are not standardized-Some are in different languages or formats.

 Also, some column names such as "order date(DateOrders)" and "shipping date(DateOrders)" are not clean or consistent,which makes them harder to use inanalysis.



2.Do you have any concerns about your dataset? 

Yes,I am concerned that the date fields are not in a proper datetime format, and the location names (states and countries) are inconsistent or messy.

This can cause errors when doing grouping,filtering, or time-based analysis.

3.Is there anything you want to make note of for the next phase of your analysis, which is cleaning data? 

For the cleaning phase, I need to:

Standardize column names so they are simple and consistent.

Handle missing values to avoid incorrect calculations.

Convert date columns into proper datetime formats for trend analysis.

Standardize geographic fields like state and country names so they match.

Remove unnecessary columns that do not add value to the analysis.